In [ ]:
import numpy as np
import pandas as pd
import pyreadr

In [ ]:
data = pyreadr.read_r("../models/r3PG/vignettes_build/vignette_data/solling.rda")
data.keys()

In [ ]:
climate = pd.DataFrame(data["climate_solling"])
climate["idx"] = np.arange(0, len(climate))

obv = pd.DataFrame(data["observ_solling"])
obv["month"] = pd.to_datetime(obv["date"]).dt.month
obv["year"] = pd.to_datetime(obv["date"]).dt.year

obv = obv.merge(climate, on=["month", "year"])[
    [
        "idx",
        "month",
        "year",
        "date",
        "biom_stem",
        "biom_foliage",
        "biom_root",
        "basal_area",
        "stems_n",
        "dbh",
        "height",
    ]
]

obv = obv.rename(
    columns={
        "dbh": "DBH",
        "biom_stem": "WS",
        "biom_foliage": "WF",
        "biom_root": "WR",
        "basal_area": "BA",
        "stems_n": "N",
        "height": "Height",
    }
)

In [ ]:
param_df = pd.DataFrame(data["param_solling"])

params_bounds = {}
for _, row in param_df.iterrows():
    if pd.notna(row["min"]) and pd.notna(row["max"]):
        params_bounds[row["param_name"]] = (row["min"], row["max"])

In [ ]:
param_default = pd.read_excel("../data/data.default.xlsx")

params = pd.DataFrame(data["param_solling"])
params = params.rename(columns={"param_name": "parameter", "default": "piab"})
params = params[["parameter", "piab"]]
params["piab"] = params["piab"].fillna(param_default["default"])

params.head()

In [ ]:
with pd.ExcelWriter("../data/solling_data.xlsx", engine="openpyxl") as writer:
    data["climate_solling"].to_excel(writer, sheet_name="climate", index=False)
    data["site_solling"].to_excel(writer, sheet_name="site", index=False)
    data["species_solling"].to_excel(writer, sheet_name="species", index=False)
    params.to_excel(writer, sheet_name="parameters", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="thinning", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="sizeDist", index=False)
    obv.to_excel(writer, sheet_name="observed", index=False)
    data["param_solling"].to_excel(writer, sheet_name="param_bound", index=False)

# Davos data processing

In [ ]:
import polars as pl

davos_df = pl.read_csv("../data/davos_meteo_data_site_id_36.csv")

davos_df = davos_df.with_columns(pl.col("ts").str.to_datetime("%Y-%m-%d"))
davos_df = davos_df.with_columns(
    [pl.col("ts").dt.month().alias("month"), pl.col("ts").dt.year().alias("year")]
)

davos_df = davos_df.rename(
    {
        "tas": "tmp_ave",
        "tasmax": "tmp_max",
        "tasmin": "tmp_min",
        "pr": "prcp",
        "gh": "srad",
        "vpd": "vpd_day",
    }
)

davos_df = davos_df.select(
    ["ts", "month", "year", "tmp_ave", "tmp_max", "tmp_min", "prcp", "srad", "vpd_day"]
)

davos_df = (
    davos_df.group_by(["month", "year"])
    .agg(
        pl.col("tmp_ave").mean().alias("tmp_ave"),
        pl.col("tmp_max").mean().alias("tmp_max"),
        pl.col("tmp_min").mean().alias("tmp_min"),
        pl.col("prcp").sum().alias("prcp"),
        pl.col("srad").mean().alias("srad"),
        pl.col("vpd_day").mean().alias("vpd_day"),
        (pl.col("tmp_ave") < 0.0).sum().alias("frost_days"),
    )
    .sort(["year", "month"])
)

davos_df

In [ ]:
import pandas as pd
from docx import Document

param_default = pd.read_excel("../data/data.default.xlsx")


def docx_tables_to_dfs(file_path):
    """Extract all tables from a .docx file and return a list of DataFrames."""
    doc = Document(file_path)
    all_dfs = []

    for table in doc.tables:
        data = []
        for row in table.rows:
            row_data = [cell.text.strip() for cell in row.cells]
            data.append(row_data)

        # Convert to DataFrame
        if data:
            # Assume first row is header
            df = pd.DataFrame(data[1:], columns=data[0])
            all_dfs.append(df)

    return all_dfs


# Use the function
file_path = "../data/gcb15011-sup-0001-supinfo-2.docx"
dfs = docx_tables_to_dfs(file_path)

# Species: P. abies
param_df = dfs[2]
param_df.head()
parameter_df = pd.DataFrame(param_df.values[1:], columns=param_df.values[0])[
    ["Parameter", "50.00%"]
]
parameter_df.rename(columns={"50.00%": "piab", "Parameter": "parameter"}, inplace=True)

parameter_df = pd.merge(param_default, parameter_df, on="parameter", how="left")
parameter_df["piab"] = parameter_df["piab"].fillna(parameter_df["default"])

parameter_df = parameter_df[["parameter", "piab"]]

In [ ]:
# Site Data

site_df = {
    "latitude": 46.802,
    "altitude": 1650,
    "soil_class": 3,
    "asw_i": 999,
    "asw_min": 0,
    "asw_max": 106,
    "from": "1981-01",
    "to": "2025-12",
}

species_df = {
    "species": "piab",
    "planted": "1800-01",
    "fertility": 0.5,
    "stem_n": 830,
}

In [ ]:
with pd.ExcelWriter("../data/davos_data.xlsx", engine="openpyxl") as writer:
    davos_df.to_pandas().to_excel(writer, sheet_name="climate", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="thinning", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="sizeDist", index=False)
    parameter_df.to_excel(writer, sheet_name="parameters", index=False)